In [1]:
!pip install pyspark

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, initcap, when, count, isnan, to_date

spark = SparkSession.builder \
    .appName("SalesDataLake_Bronze_to_Silver") \
    .getOrCreate()

bronze_path = "data_lake/bronze/sales.csv"
df_bronze = spark.read.csv(bronze_path, header=True, inferSchema=True)

print("Dataset Schema:")
df_bronze.printSchema()
print("First 5 rows:")
df_bronze.show(5)

original_count = df_bronze.count()
print(f"Total Original Records: {original_count}")

duplicate_count = original_count - df_bronze.dropDuplicates().count()
print(f"Number of exact duplicate rows: {duplicate_count}")

print("Count of Null/NaN values per column:")
null_exprs = []
for c, dtype in df_bronze.dtypes:
    if dtype in ('double', 'float'):
        null_exprs.append(count(when(col(c).isNull() | isnan(col(c)), c)).alias(c))
    else:
        null_exprs.append(count(when(col(c).isNull(), c)).alias(c))

df_bronze.select(null_exprs).show()

Dataset Schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

First 5 rows:
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+------------

In [13]:
from pyspark.sql.functions import col, trim, initcap, udf
from pyspark.sql.types import DateType
from datetime import datetime

df_silver = df_bronze.dropDuplicates()
text_columns = [f.name for f in df_silver.schema.fields if f.dataType.typeName() == 'string']
for c in text_columns:
    df_silver = df_silver.withColumn(c, initcap(trim(col(c))))

df_silver = df_silver.dropna(subset=["order_id"])

df_silver = df_silver.na.fill(0, subset=["quantity", "unit_price", "discount_percent"])

df_silver = df_silver.filter(col("quantity") >= 0)
df_silver = df_silver.filter(col("unit_price") >= 0)
df_silver = df_silver.filter((col("discount_percent") >= 0) & (col("discount_percent") <= 100))

def safe_parse_date(date_str):
    if not date_str:
        return None
    formats = ['%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y']
    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt).date()
        except ValueError:
            continue
    return None

safe_date_udf = udf(safe_parse_date, DateType())

df_silver = df_silver.withColumn("order_date", safe_date_udf(col("order_date")))

df_silver = df_silver.filter(col("order_date").isNotNull())

In [14]:
import os
import glob
import shutil

final_count = df_silver.count()
print(f"Row count BEFORE preprocessing: {original_count}")
print(f"Row count AFTER preprocessing: {final_count}")

temp_silver_dir = "data_lake/silver/temp_sales"
df_silver.coalesce(1).write.mode("overwrite").csv(temp_silver_dir, header=True)

part_file = glob.glob(f"{temp_silver_dir}/part-*.csv")[0]
final_silver_path = "data_lake/silver/sales_silver.csv"
os.rename(part_file, final_silver_path)

shutil.rmtree(temp_silver_dir)
print(f"Successfully saved as exactly: {final_silver_path}")
df_silver_read = spark.read.csv(final_silver_path, header=True, inferSchema=True)
print("\nVerifying saved Silver Data:")
df_silver_read.show(5)

print("""
Preprocessing Summary:
1. Removed exact duplicate rows.
2. Handled missing values by dropping rows without IDs and filling missing numerics with 0.
3. Trimmed trailing/leading spaces and standardized capitalization for all text fields.
4. Filtered out logically invalid numerical values (e.g., negative prices or quantities).
5. Standardized the date column and removed unparseable date values.
""")

Row count BEFORE preprocessing: 1000
Row count AFTER preprocessing: 966
Successfully saved as exactly: data_lake/silver/sales_silver.csv

Verifying saved Silver Data:
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|     customer_name|             product| category|quantity|unit_price|discount_percent|  payment_method|      city|      state|order_date|order_status|
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|  100331|      C0005|      Sakshi Joshi|            Football|   Sports|       2|       670|              30|     Credit Card| Bengaluru|  Karnataka|2025-06-06|   Cancelled|
|  100333|      C0263|Siddharth Kulkarni|    Programming Book|    Books|       4|       670|               0|     Net Banking|      Pune|